In [1]:
import os

# Honor an existing CUDA selection from the shell; default to GPU 1.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import argparse
import json
import pickle
import sys
from pathlib import Path

sys.modules.setdefault("learn_propagator", sys.modules[__name__])
from typing import Optional, Tuple

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optax
from jax import lax, random
from tqdm import tqdm


Array = jax.Array
PRNGKey = Array

J_zz = 1.0
omega_factor = 10
n_modes = 10
FNO_MODES = 48

flax_dense_kernel_init = jax.nn.initializers.lecun_normal()
flax_embed_init = jax.nn.initializers.variance_scaling(1.0, "fan_in", "normal", out_axis=0)

def zeros_init(shape, dtype=jnp.float32):
    return jnp.zeros(shape, dtype=dtype)

In [2]:
def lattice_visit_order(Lx: int, Ly: int, site_order: str = "row_major"):
    coords = []
    if site_order == "row_major":
        for y in range(Ly):
            for x in range(Lx):
                coords.append((x, y))
        return coords

    if site_order == "col_major":
        for x in range(Lx):
            for y in range(Ly):
                coords.append((x, y))
        return coords

    if site_order == "snake":
        if Lx >= Ly:
            for y in range(Ly):
                xs = range(Lx) if y % 2 == 0 else range(Lx - 1, -1, -1)
                for x in xs:
                    coords.append((x, y))
        else:
            for x in range(Lx):
                ys = range(Ly) if x % 2 == 0 else range(Ly - 1, -1, -1)
                for y in ys:
                    coords.append((x, y))
        return coords

    raise ValueError(f"Unknown site_order: {site_order}")


def make_lattice_bonds(Lx: int, Ly: int, site_order: str = "row_major"):
    coord_to_idx = {coord: idx for idx, coord in enumerate(lattice_visit_order(Lx, Ly, site_order))}
    bonds_i = []
    bonds_j = []
    for y_c in range(Ly):
        for x_c in range(Lx):
            i = coord_to_idx[(x_c, y_c)]
            if x_c + 1 < Lx:
                bonds_i.append(i)
                bonds_j.append(coord_to_idx[(x_c + 1, y_c)])
            if y_c + 1 < Ly:
                bonds_i.append(i)
                bonds_j.append(coord_to_idx[(x_c, y_c + 1)])
    return bonds_i, bonds_j

def generate_grf_trajectories(key, batch_size, t_steps, t_max, n_modes=n_modes):
    kx1, kx2, kx3, kz1, kz2, kz3 = random.split(key, 6)

    t = jnp.linspace(0.0, t_max, t_steps, dtype=jnp.float32)
    k = jnp.arange(1, n_modes + 1, dtype=jnp.float32)
    omega = k * (jnp.pi / t_max) * omega_factor
    amp_decay = 1.0 / (k**1.5)

    amps_x = random.uniform(kx1, (batch_size, n_modes), minval=-0.50, maxval=0.50) * amp_decay[None, :]
    phases_x = random.uniform(kx2, (batch_size, n_modes), minval=0.0, maxval=2 * jnp.pi)
    hx0 = random.uniform(kx3, (batch_size, 1), minval=-0.05, maxval=0.05) + 1.0
    args_x = omega[None, :, None] * t[None, None, :] + phases_x[:, :, None]
    sig_x = jnp.sum(amps_x[:, :, None] * jnp.sin(args_x), axis=1)
    hx = hx0 + sig_x

    amps_z = random.uniform(kz1, (batch_size, n_modes), minval=-0.05, maxval=0.05) * amp_decay[None, :]
    phases_z = random.uniform(kz2, (batch_size, n_modes), minval=0.0, maxval=2 * jnp.pi)
    hz0 = random.uniform(kz3, (batch_size, 1), minval=-0.05, maxval=0.05) + 0.00
    args_z = omega[None, :, None] * t[None, None, :] + phases_z[:, :, None]
    sig_z = jnp.sum(amps_z[:, :, None] * jnp.sin(args_z), axis=1)
    hz = hz0 + sig_z

    return jnp.stack([hx, hz], axis=-1)


def make_random_initial_ctx(seed: int, ctx_tokens: int, embed_dim: int) -> Array:
    key = random.PRNGKey(seed)
    initial_ctx = random.normal(key, (1, ctx_tokens, embed_dim), dtype=jnp.float32)
    norms = jnp.linalg.norm(initial_ctx, axis=-1, keepdims=True)
    return initial_ctx / jnp.clip(norms, a_min=1e-8)


def make_canonical_initial_ctx(ctx_tokens: int, embed_dim: int) -> Array:
    if ctx_tokens > embed_dim:
        raise ValueError(
            f"Cannot build canonical M0 with ctx_tokens={ctx_tokens} "
            f"and embed_dim={embed_dim}: need ctx_tokens <= embed_dim."
        )
    initial_ctx = jnp.zeros((1, ctx_tokens, embed_dim), dtype=jnp.float32)
    token_ids = jnp.arange(ctx_tokens, dtype=jnp.int32)
    return initial_ctx.at[0, token_ids, token_ids].set(1.0)

def make_apply_H_exact(bonds_i_np, bonds_j_np, n_spins: int, j_zz: float = J_zz):
    bond_pairs = tuple((int(bi), int(bj)) for bi, bj in zip(bonds_i_np, bonds_j_np))

    @jax.jit
    def apply_H_exact(psi: Array, h_x_val: Array, h_z_val: Array):
        dim = psi.shape[0]
        idx = jnp.arange(dim, dtype=jnp.int32)

        e_diag = jnp.zeros((dim,), dtype=jnp.float32)
        z_sum = jnp.zeros((dim,), dtype=jnp.float32)

        for bi, bj in bond_pairs:
            sbi = (idx >> bi) & 1
            sbj = (idx >> bj) & 1
            zi = (1 - 2 * sbi).astype(jnp.float32)
            zj = (1 - 2 * sbj).astype(jnp.float32)
            e_diag = e_diag + (-j_zz) * (zi * zj)

        for i in range(n_spins):
            bi = (idx >> i) & 1
            zi = (1 - 2 * bi).astype(jnp.float32)
            z_sum = z_sum + zi

        e_diag = e_diag + (-h_z_val) * z_sum
        res = e_diag.astype(psi.real.dtype) * psi

        psi_tensor = psi.reshape((2,) * n_spins)
        for i in range(n_spins):
            flipped = jnp.flip(psi_tensor, axis=n_spins - 1 - i).reshape(-1)
            res = res + (-h_x_val) * flipped

        return res

    return apply_H_exact

In [3]:
class Linear(eqx.Module):
    weight: Array
    bias: Optional[Array]
    use_bias: bool = eqx.field(static=True)

    def __init__(
        self,
        in_features: int,
        out_features: int,
        *,
        key: PRNGKey,
        use_bias: bool = True,
        kernel_init=flax_dense_kernel_init,
        bias_init=zeros_init,
        dtype=jnp.float32,
    ):
        wkey, bkey = random.split(key)
        self.weight = kernel_init(wkey, (in_features, out_features), dtype).T
        self.use_bias = use_bias
        self.bias = bias_init((out_features,), dtype) if use_bias else None

    def __call__(self, x: Array) -> Array:
        y = jnp.matmul(x, self.weight.T)
        if self.use_bias and self.bias is not None:
            y = y + self.bias
        return y


class Embedding(eqx.Module):
    weight: Array

    def __init__(self, num_embeddings: int, embedding_size: int, *, key: PRNGKey, dtype=jnp.float32):
        self.weight = flax_embed_init(key, (num_embeddings, embedding_size), dtype)

    def __call__(self, idx: Array) -> Array:
        return self.weight[idx]


class LayerNorm(eqx.Module):
    weight: Array
    bias: Array
    eps: float = eqx.field(static=True)

    def __init__(self, dim: int, eps: float = 1e-6, *, key: Optional[PRNGKey] = None, dtype=jnp.float32):
        del key
        self.weight = jnp.ones((dim,), dtype=dtype)
        self.bias = jnp.zeros((dim,), dtype=dtype)
        self.eps = eps

    def __call__(self, x: Array) -> Array:
        orig_dtype = x.dtype
        dtype = jnp.result_type(x.dtype, jnp.float32)
        y = x.astype(dtype)
        mean = jnp.mean(y, axis=-1, keepdims=True)
        var = jnp.maximum(jnp.mean(y * y, axis=-1, keepdims=True) - mean * mean, 0.0)
        y = (y - mean) * lax.rsqrt(var + self.eps)
        y = y * self.weight.astype(dtype) + self.bias.astype(dtype)
        return y.astype(orig_dtype)


class MultiHeadDotProductAttention(eqx.Module):
    query_proj: Linear
    key_proj: Linear
    value_proj: Linear
    out_proj: Linear
    num_heads: int = eqx.field(static=True)
    embed_dim: int = eqx.field(static=True)
    head_dim: int = eqx.field(static=True)

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        *,
        key: PRNGKey,
    ):
        if embed_dim % num_heads != 0:
            raise ValueError(f"embed_dim={embed_dim} must be divisible by num_heads={num_heads}")
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        kq, kk, kv, ko = random.split(key, 4)
        self.query_proj = Linear(embed_dim, embed_dim, key=kq)
        self.key_proj = Linear(embed_dim, embed_dim, key=kk)
        self.value_proj = Linear(embed_dim, embed_dim, key=kv)
        self.out_proj = Linear(embed_dim, embed_dim, key=ko)

    def __call__(self, query: Array, key: Array, value: Array, mask: Optional[Array] = None) -> Array:
        q = self.project_query(query)
        k, v = self.project_kv(key, value)
        return self.attend_projected(q, k, v, mask=mask)

    def project_query(self, query: Array) -> Array:
        B, Lq, _ = query.shape
        q = self.query_proj(query).reshape(B, Lq, self.num_heads, self.head_dim)
        return jnp.transpose(q, (0, 2, 1, 3))

    def project_kv(self, key: Array, value: Array) -> Tuple[Array, Array]:
        B, Lk, _ = key.shape
        k = self.key_proj(key).reshape(B, Lk, self.num_heads, self.head_dim)
        v = self.value_proj(value).reshape(B, Lk, self.num_heads, self.head_dim)
        return jnp.transpose(k, (0, 2, 1, 3)), jnp.transpose(v, (0, 2, 1, 3))

    def attend_projected(self, q: Array, k: Array, v: Array, mask: Optional[Array] = None) -> Array:
        B, _, Lq, _ = q.shape
        logits = jnp.matmul(q, jnp.swapaxes(k, -1, -2)) * (self.head_dim ** -0.5)
        if mask is not None:
            if mask.ndim == 2:
                logits = jnp.where(mask[None, None, :, :], logits, jnp.finfo(logits.dtype).min)
            elif mask.ndim == 3:
                logits = jnp.where(mask[:, None, :, :], logits, jnp.finfo(logits.dtype).min)
            else:
                raise ValueError(f"Unsupported mask rank: {mask.ndim}")
        attn = jax.nn.softmax(logits, axis=-1)
        out = jnp.matmul(attn, v)
        out = jnp.transpose(out, (0, 2, 1, 3)).reshape(B, Lq, self.embed_dim)
        return self.out_proj(out)

class SpectralConv1D(eqx.Module):
    weight_re: Array
    weight_im: Array
    in_channels: int = eqx.field(static=True)
    out_channels: int = eqx.field(static=True)
    modes: int = eqx.field(static=True)

    def __init__(self, in_channels: int, out_channels: int, modes: int, *, key: PRNGKey):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes
        k1, k2 = random.split(key)
        scale = 0.02
        self.weight_re = scale * random.normal(k1, (modes, in_channels, out_channels), dtype=jnp.float32)
        self.weight_im = scale * random.normal(k2, (modes, in_channels, out_channels), dtype=jnp.float32)

    def __call__(self, x: Array) -> Array:
        B, T, _ = x.shape
        x_ft = jnp.fft.rfft(x, axis=1)
        t_freq = x_ft.shape[1]
        m = min(self.modes, t_freq)

        w = self.weight_re[:m] + 1j * self.weight_im[:m]
        out_ft_low = jnp.swapaxes(jnp.matmul(jnp.swapaxes(x_ft[:, :m, :], 0, 1), w), 0, 1)
        pad = jnp.zeros((B, t_freq - m, self.out_channels), dtype=out_ft_low.dtype)
        out_ft = jnp.concatenate([out_ft_low, pad], axis=1)
        return jnp.fft.irfft(out_ft, n=T, axis=1)


class FNO1D(eqx.Module):
    lift: Linear
    spectral_layers: Tuple[SpectralConv1D, ...]
    linear_layers: Tuple[Linear, ...]
    norm: LayerNorm
    proj: Linear
    out_tokens: int = eqx.field(static=True)
    out_dim: int = eqx.field(static=True)
    in_fields: int = eqx.field(static=True)
    depth: int = eqx.field(static=True)

    def __init__(
        self,
        modes: int,
        width: int,
        out_tokens: int,
        out_dim: int,
        in_fields: int,
        depth: int = 4,
        *,
        key: PRNGKey,
    ):
        self.out_tokens = out_tokens
        self.out_dim = out_dim
        self.in_fields = in_fields
        self.depth = depth

        keys = random.split(key, 2 * depth + 3)
        self.lift = Linear(in_fields + 1, width, key=keys[0])
        self.spectral_layers = tuple(
            SpectralConv1D(width, width, modes, key=keys[1 + i]) for i in range(depth)
        )
        self.linear_layers = tuple(
            Linear(width, width, key=keys[1 + depth + i]) for i in range(depth)
        )
        self.norm = LayerNorm(width)
        self.proj = Linear(width, out_tokens * out_dim, key=keys[-1])

    def __call__(self, h_fields: Array) -> Array:
        B, T, _ = h_fields.shape
        t = jnp.linspace(0, 1, T)[:, jnp.newaxis]
        t_grid = jnp.broadcast_to(t[None, :, :], (B, T, 1))
        x = jnp.concatenate([h_fields, t_grid], axis=-1)
        x = self.lift(x)

        for spectral_layer, linear_layer in zip(self.spectral_layers, self.linear_layers):
            x = jax.nn.gelu(spectral_layer(x) + linear_layer(x))

        x = self.norm(x)
        x = self.proj(x)
        return x.reshape(B, T, self.out_tokens, self.out_dim)


def causal_mask(seq_len: int) -> Array:
    return jnp.tril(jnp.ones((seq_len, seq_len), dtype=bool))


class SpinDecoderBlock(eqx.Module):
    num_heads: int = eqx.field(static=True)
    embed_dim: int = eqx.field(static=True)
    mlp_mult: int = eqx.field(static=True)
    ln1: LayerNorm
    ln2: LayerNorm
    ln3: LayerNorm
    self_attn: MultiHeadDotProductAttention
    cross_attn: MultiHeadDotProductAttention
    mlp_in: Linear
    mlp_out: Linear

    def __init__(
        self,
        num_heads: int,
        embed_dim: int,
        mlp_mult: int = 4,
        *,
        key: PRNGKey,
    ):
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.mlp_mult = mlp_mult
        k1, k2, k3, k4 = random.split(key, 4)
        self.ln1 = LayerNorm(embed_dim)
        self.ln2 = LayerNorm(embed_dim)
        self.ln3 = LayerNorm(embed_dim)
        self.self_attn = MultiHeadDotProductAttention(embed_dim, num_heads, key=k1)
        self.cross_attn = MultiHeadDotProductAttention(embed_dim, num_heads, key=k2)
        self.mlp_in = Linear(embed_dim, embed_dim * mlp_mult, key=k3)
        self.mlp_out = Linear(embed_dim * mlp_mult, embed_dim, key=k4)

    def __call__(self, x: Array, ctx: Array, mask: Array) -> Array:
        y = self.ln1(x)
        y = self.self_attn(y, y, y, mask=mask)
        x = x + y

        y = self.ln2(x)
        y = self.cross_attn(y, ctx, ctx)
        x = x + y

        y = self.ln3(x)
        y = jax.nn.gelu(self.mlp_in(y))
        y = self.mlp_out(y)
        x = x + y
        return x




In [4]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser()
    fix_group = parser.add_mutually_exclusive_group()
    fix_group.add_argument(
        "--fix-M0",
        dest="fix_M0",
        action="store_true",
        help="Use the default fixed random unit-norm M0 tokens as the initial context for integrating Mdot(t).",
    )
    fix_group.add_argument(
        "--canonical-M0",
        dest="canonical_M0",
        action="store_true",
        help="Use fixed canonical-basis M0 tokens as the initial context for integrating Mdot(t).",
    )
    fix_group.add_argument(
        "--diff-M",
        dest="diff_M",
        action="store_true",
        help="Use a learnable shared M0 instead of the default fixed M0.",
    )
    parser.add_argument(
        "--resume",
        type=str,
        default=None,
        help="Resume from a checkpoint written by this script. Params-only pickles are also accepted.",
    )
    parser.add_argument(
        "--checkpoint-path",
        type=str,
        default="learn_propagator_latest_checkpoint_4x4.pkl",
        help="Path where the latest resumable training checkpoint is written.",
    )
    parser.add_argument("--Lx", type=int, default=4)
    parser.add_argument("--Ly", type=int, default=4)
    parser.add_argument("--site-order", type=str, default="snake", choices=("row_major", "col_major", "snake"))
    parser.add_argument("--t-steps", type=int, default=200)
    parser.add_argument("--t-max", type=float, default=0.5)
    parser.add_argument("--batch-size", type=int, default=6)
    parser.add_argument("--num-time-points", type=int, default=4)
    parser.add_argument("--num-mc-samples", type=int, default=128)
    parser.add_argument("--d-model", type=int, default=96)
    parser.add_argument("--fno-width", type=int, default=128)
    parser.add_argument("--num-layers", type=int, default=3)
    parser.add_argument("--mlp-mult", type=int, default=4)
    parser.add_argument("--ctx-tokens", type=int, default=4)
    parser.add_argument("--integration-rule", type=str, default="simpson", choices=("trap", "simpson"))
    parser.add_argument("--anchor-weight", type=float, default=1.0)
    parser.add_argument("--score-function-weight", type=float, default=1.0)
    parser.add_argument("--start-lr", type=float, default=5e-4)
    parser.add_argument("--min-lr", type=float, default=5e-6)
    parser.add_argument("--decay-rate", type=float, default=0.95)
    parser.add_argument("--decay-steps", type=int, default=2000)
    parser.add_argument("--phase-mode", type=str, default="raw", choices=("raw", "softsign"))
    parser.add_argument(
        "--gauge-centering",
        type=str,
        default="per_time",
        choices=("global", "per_time"),
        help="Center the propagator residual globally over all sampled points or separately for each sampled time.",
    )
    parser.add_argument("--self-kv-cache", action="store_true")
    parser.add_argument("--steps", type=int, default=120000)
    parser.add_argument("--anchor-pretrain-steps", type=int, default=500)
    parser.add_argument("--benchmark-every", type=int, default=5000)
    parser.add_argument("--benchmark-key", type=int, default=14850)
    parser.add_argument(
        "--time-curriculum",
        action="store_true",
        help="Use a sampled-time curriculum: early TDVP steps sample only early times, then expand to the full trajectory.",
    )
    parser.add_argument(
        "--time-curriculum-start-frac",
        type=float,
        default=0.5,
        help="Initial fraction of the time grid exposed to the TDVP loss when --time-curriculum is enabled.",
    )
    parser.add_argument(
        "--time-curriculum-power",
        type=float,
        default=1.0,
        help="Power-law schedule for widening the sampled-time window. 1.0 is linear; >1 grows slower early.",
    )
    parser.add_argument(
        "--state-benchmark-count",
        type=int,
        default=0,
        help="Number of random product-state initial conditions to benchmark against exact evolution after training.",
    )
    parser.add_argument(
        "--state-benchmark-basis",
        type=str,
        default="z",
        choices=("z", "x", "both"),
        help="Sample benchmark product states from the Z basis, X basis, or both.",
    )
    parser.add_argument(
        "--state-benchmark-seed",
        type=int,
        default=20260419,
        help="PRNG seed used to sample benchmark product-state bitstrings.",
    )
    parser.add_argument(
        "--state-benchmark-alpha-chunk",
        type=int,
        default=8192,
        help="Alpha-state chunk size when reconstructing benchmark wavefunctions from propagator amplitudes.",
    )
    parser.add_argument(
        "--state-benchmark-beta-chunk",
        type=int,
        default=1024,
        help="Beta-state chunk size for X-basis benchmarks, which sum over Z-basis columns.",
    )
    parser.add_argument(
        "--state-benchmark-max-x-spins",
        type=int,
        default=10,
        help="Skip X-basis state benchmarks when n_spins exceeds this threshold, since they require all Z-basis columns.",
    )
    parser.add_argument(
        "--state-benchmark-fixed-basis",
        type=str,
        default=None,
        choices=("z", "x"),
        help="Benchmark one fixed initial product state instead of random samples.",
    )
    parser.add_argument(
        "--state-benchmark-fixed-bits",
        type=str,
        default=None,
        help="Bitstring for the fixed state benchmark, e.g. 00001111. Required with --state-benchmark-fixed-basis.",
    )
    parser.add_argument("--print-every", type=int, default=50)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output-dir", type=str, default="learn_propagator_run")
    return parser





def integrate_context_velocity(initial_ctx: Array, mdot: Array, dt: float, integration_rule: str) -> Array:
    if mdot.ndim != 4:
        raise ValueError(f"Expected mdot with shape (B, T, C, D), got {mdot.shape}")
    if mdot.shape[1] == 0:
        raise ValueError("Cannot integrate an empty time axis for mdot.")
    m0 = initial_ctx[:, None, :, :]
    if mdot.shape[1] == 1:
        return m0

    if integration_rule == "trap":
        increments = 0.5 * dt * (mdot[:, 1:, :, :] + mdot[:, :-1, :, :])
        cumulative = jnp.cumsum(increments, axis=1)
        return jnp.concatenate([m0, m0 + cumulative], axis=1)

    if integration_rule != "simpson":
        raise ValueError(f"Unknown integration rule: {integration_rule}")

    flat_mdot = mdot.reshape(mdot.shape[0], mdot.shape[1], -1)
    cumulative_flat = jnp.zeros_like(flat_mdot)
    if mdot.shape[1] == 2:
        cumulative_flat = cumulative_flat.at[:, 1].set(0.5 * dt * (flat_mdot[:, 0] + flat_mdot[:, 1]))
    else:
        first_step = dt * (5.0 * flat_mdot[:, 0] + 8.0 * flat_mdot[:, 1] - flat_mdot[:, 2]) / 12.0
        cumulative_flat = cumulative_flat.at[:, 1].set(first_step)
        for i in range(2, mdot.shape[1]):
            if i % 2 == 0:
                step_val = cumulative_flat[:, i - 2] + dt * (
                    flat_mdot[:, i - 2] + 4.0 * flat_mdot[:, i - 1] + flat_mdot[:, i]
                ) / 3.0
            else:
                step_val = cumulative_flat[:, i - 1] + dt * (
                    -flat_mdot[:, i - 2] + 8.0 * flat_mdot[:, i - 1] + 5.0 * flat_mdot[:, i]
                ) / 12.0
            cumulative_flat = cumulative_flat.at[:, i].set(step_val)
    return m0 + cumulative_flat.reshape(mdot.shape)


def arrays_only(model):
    return eqx.filter(model, eqx.is_array)




def combine_model(arrays, template):
    static = eqx.partition(template, eqx.is_array)[1]
    return eqx.combine(arrays, static)


def upgrade_resume_model(model_arrays, template):
    """Upgrade older checkpoint trees to the current PropagatorNOQS layout.

    Older checkpoints may miss newer static metadata such as `mlp_mult`.
    We rebuild the lightweight arrays-only Equinox module into the current
    class layout while preserving all stored leaves and adopting the current
    run's `dt`.
    """
    if not isinstance(model_arrays, PropagatorNOQS):
        return model_arrays

    upgraded = PropagatorNOQS.__new__(PropagatorNOQS)
    for name in (
        "embed_dim",
        "num_heads",
        "num_layers",
        "mlp_mult",
        "fno_modes",
        "fno_width",
        "ctx_tokens",
        "in_fields",
        "phase_mode",
        "self_kv_cache",
        "fixed_M0_mode",
        "fixed_M0_seed",
        "n_spins",
        "integration_rule",
        "dt",
        "fno",
        "token_embed",
        "pos_embed",
        "start_token",
        "initial_ctx",
        "blocks",
        "ln_final",
        "amp_head",
        "phase_head",
    ):
        if name == "dt":
            value = float(template.dt)
        elif hasattr(model_arrays, name):
            value = getattr(model_arrays, name)
        else:
            value = getattr(template, name)
        object.__setattr__(upgraded, name, value)
    return upgraded


def upgrade_resume_model_in_tree(tree, template):
    """Upgrade any PropagatorNOQS pytrees nested inside optimizer state."""

    def _fix(x):
        if isinstance(x, PropagatorNOQS):
            return upgrade_resume_model(x, template)
        return x

    return jax.tree_util.tree_map(
        _fix,
        tree,
        is_leaf=lambda x: isinstance(x, PropagatorNOQS),
    )




def save_training_checkpoint(
    path: Path,
    *,
    step: int,
    model,
    opt_state,
    rng,
    best_loss: float,
    best_model,
    loss_hist,
    config: dict,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "format": "learn_propagator_checkpoint_v1",
        "step": int(step),
        "model": jax.device_get(arrays_only(model)),
        "opt_state": jax.device_get(opt_state),
        "rng": np.asarray(jax.device_get(rng)),
        "best_loss": float(best_loss),
        "best_model": jax.device_get(arrays_only(best_model)),
        "loss_hist": [float(x) for x in loss_hist],
        "config": config,
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f)


def load_resume_payload(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))




def sampled_time_curriculum_fraction(args: argparse.Namespace, step: int) -> float:
    if not args.time_curriculum:
        return 1.0
    if args.steps <= 1:
        progress = 1.0
    else:
        progress = min(max(float(step) / float(args.steps - 1), 0.0), 1.0)
    return float(args.time_curriculum_start_frac) + (
        1.0 - float(args.time_curriculum_start_frac)
    ) * (progress ** float(args.time_curriculum_power))


def sampled_time_curriculum_max_t(args: argparse.Namespace, step: int) -> int:
    frac = sampled_time_curriculum_fraction(args, step)
    return max(1, min(int(args.t_steps), int(np.ceil(frac * int(args.t_steps)))))


def save_benchmark_series(output_dir: Path, step: int, result: dict) -> None:
    series = result["series"]
    np.savez(
        output_dir / f"benchmark_step_{step:05d}.npz",
        operator_fidelity=np.asarray(series["operator_fidelity"], dtype=np.float32),
        column_fidelity=np.asarray(series["column_fidelity"], dtype=np.float32),
        fro_error=np.asarray(series["fro_error"], dtype=np.float32),
    )




def sigma_to_alpha_bits(sigmas_idx: Array) -> Array:
    return (sigmas_idx // 2).astype(jnp.int32)


def sigma_flip_alpha(sigmas_idx: Array, site: int) -> Array:
    flipped = jnp.bitwise_xor(sigmas_idx[:, site], jnp.int32(2))
    return sigmas_idx.at[:, site].set(flipped)


def sample_identity_on_support(key: PRNGKey, batch_states: int, n_samples: int, n_spins: int) -> Array:
    bern = random.bernoulli(key, 0.5, (batch_states, n_samples, n_spins))
    return 3 * bern.astype(jnp.int32)


def sample_identity_off_support(key: PRNGKey, on_support: Array) -> Array:
    batch_states, n_samples, n_spins = on_support.shape
    sites = random.randint(key, (batch_states, n_samples), 0, n_spins)
    batch_idx = jnp.arange(batch_states)[:, None]
    sample_idx = jnp.arange(n_samples)[None, :]
    chosen = on_support[batch_idx, sample_idx, sites]
    off = jnp.bitwise_xor(chosen, jnp.int32(1))
    return on_support.at[batch_idx, sample_idx, sites].set(off)




class PropagatorNOQS(eqx.Module):
    embed_dim: int = eqx.field(static=True)
    num_heads: int = eqx.field(static=True)
    num_layers: int = eqx.field(static=True)
    mlp_mult: int = eqx.field(static=True)
    fno_modes: int = eqx.field(static=True)
    fno_width: int = eqx.field(static=True)
    ctx_tokens: int = eqx.field(static=True)
    in_fields: int = eqx.field(static=True)
    phase_mode: str = eqx.field(static=True)
    self_kv_cache: bool = eqx.field(static=True)
    fixed_M0_mode: Optional[str] = eqx.field(static=True)
    fixed_M0_seed: Optional[int] = eqx.field(static=True)
    n_spins: int = eqx.field(static=True)
    integration_rule: str = eqx.field(static=True)
    dt: float = eqx.field(static=True)

    fno: object
    token_embed: object
    pos_embed: object
    start_token: Array
    initial_ctx: Optional[Array]
    blocks: Tuple[object, ...]
    ln_final: object
    amp_head: object
    phase_head: object

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        num_layers: int,
        mlp_mult: int,
        fno_modes: int,
        fno_width: int,
        ctx_tokens: int,
        in_fields: int,
        *,
        phase_mode: str = "softsign",
        self_kv_cache: bool = False,
        fixed_M0_mode: Optional[str] = None,
        fixed_M0_seed: Optional[int] = None,
        n_spins: int,
        integration_rule: str,
        dt: float,
        key: PRNGKey,
    ):
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.mlp_mult = mlp_mult
        self.fno_modes = fno_modes
        self.fno_width = fno_width
        self.ctx_tokens = ctx_tokens
        self.in_fields = in_fields
        self.phase_mode = phase_mode
        self.self_kv_cache = self_kv_cache
        self.fixed_M0_mode = fixed_M0_mode
        self.fixed_M0_seed = fixed_M0_seed
        self.n_spins = n_spins
        self.integration_rule = integration_rule
        self.dt = dt

        keys = random.split(key, num_layers + 7)
        self.fno = FNO1D(fno_modes, fno_width, ctx_tokens, embed_dim, in_fields, key=keys[0])
        self.token_embed = Embedding(4, embed_dim, key=keys[1])
        self.pos_embed = Embedding(n_spins, embed_dim, key=keys[2])
        self.start_token = 0.02 * random.normal(keys[3], (1, 1, embed_dim), dtype=jnp.float32)

        if fixed_M0_mode is None or fixed_M0_mode == "learnable_diff":
            self.initial_ctx = 0.02 * random.normal(keys[4], (1, ctx_tokens, embed_dim), dtype=jnp.float32)
        elif fixed_M0_mode == "random_unit_rows":
            self.initial_ctx = make_random_initial_ctx(
                fixed_M0_seed if fixed_M0_seed is not None else 0,
                ctx_tokens,
                embed_dim,
            )
        elif fixed_M0_mode == "canonical_basis":
            self.initial_ctx = make_canonical_initial_ctx(ctx_tokens, embed_dim)
        else:
            raise ValueError(f"Unknown fixed_M0_mode: {fixed_M0_mode}")

        self.blocks = tuple(
            SpinDecoderBlock(num_heads, embed_dim, mlp_mult=mlp_mult, key=keys[5 + i]) for i in range(num_layers)
        )
        self.ln_final = LayerNorm(embed_dim)
        self.amp_head = Linear(embed_dim, 4, key=keys[-2], kernel_init=jax.nn.initializers.orthogonal())
        self.phase_head = Linear(embed_dim, 4, key=keys[-1], kernel_init=jax.nn.initializers.normal(0.02))

    def _runtime_initial_ctx(self) -> Array:
        if self.fixed_M0_mode is None or self.fixed_M0_mode == "learnable_diff":
            return self.initial_ctx
        return lax.stop_gradient(self.initial_ctx)

    def encode_field(self, h_fields: Array) -> Tuple[Array, Array]:
        bsz, _, _ = h_fields.shape
        mdot = self.fno(h_fields)
        m0 = jnp.broadcast_to(self._runtime_initial_ctx(), (bsz, self.ctx_tokens, self.embed_dim))
        m = integrate_context_velocity(m0, mdot, self.dt, self.integration_rule)
        return m, mdot

    def get_initial_ctx(self) -> Array:
        return self._runtime_initial_ctx()

    def _decode(self, sigmas_idx: Array, ctx_tokens: Array) -> Array:
        bsz, n_sites = sigmas_idx.shape
        pos = self.pos_embed(jnp.arange(n_sites, dtype=jnp.int32))[None, :, :]
        start = jnp.tile(self.start_token, (bsz, 1, 1)) + pos[:, 0:1, :]
        prev = self.token_embed(sigmas_idx[:, :-1]) + pos[:, 1:, :]
        x = jnp.concatenate([start, prev], axis=1)
        mask = causal_mask(n_sites)
        for blk in self.blocks:
            x = blk(x, ctx_tokens, mask)
        return self.ln_final(x)

    def build_cross_kv_cache(self, ctx_tokens: Array) -> Tuple[Tuple[Array, Array], ...]:
        return tuple(blk.cross_attn.project_kv(ctx_tokens, ctx_tokens) for blk in self.blocks)

    def empty_self_kv_cache(self, batch_size: int, n_spins: int) -> Tuple[Tuple[Array, Array], ...]:
        head_dim = self.embed_dim // self.num_heads
        shape = (batch_size, self.num_heads, n_spins, head_dim)
        dtype = self.start_token.dtype
        return tuple((jnp.zeros(shape, dtype=dtype), jnp.zeros(shape, dtype=dtype)) for _ in self.blocks)

    def _decode_token_step(
        self,
        token_x: Array,
        self_kv_cache: Tuple[Tuple[Array, Array], ...],
        cross_kv_cache: Tuple[Tuple[Array, Array], ...],
        step_idx: Array,
        n_spins: int,
    ) -> Tuple[Array, Tuple[Tuple[Array, Array], ...]]:
        valid_mask = (jnp.arange(n_spins, dtype=jnp.int32) <= step_idx)[None, :]
        x = token_x
        new_self_kv_cache = []
        update_idx = (0, 0, step_idx, 0)

        for blk, cross_kv, self_kv in zip(self.blocks, cross_kv_cache, self_kv_cache):
            k_cache, v_cache = self_kv

            y = blk.ln1(x)
            q = blk.self_attn.project_query(y)
            k_new, v_new = blk.self_attn.project_kv(y, y)
            k_cache = lax.dynamic_update_slice(k_cache, k_new, update_idx)
            v_cache = lax.dynamic_update_slice(v_cache, v_new, update_idx)
            y = blk.self_attn.attend_projected(q, k_cache, v_cache, mask=valid_mask)
            x = x + y

            y = blk.ln2(x)
            q = blk.cross_attn.project_query(y)
            y = blk.cross_attn.attend_projected(q, cross_kv[0], cross_kv[1])
            x = x + y

            y = blk.ln3(x)
            y = jax.nn.gelu(blk.mlp_in(y))
            y = blk.mlp_out(y)
            x = x + y
            new_self_kv_cache.append((k_cache, v_cache))

        return x, tuple(new_self_kv_cache)

    def logits_from_tokens(self, sigmas_idx: Array, ctx_tokens: Array) -> Array:
        return self.amp_head(self._decode(sigmas_idx, ctx_tokens))

    def log_psi_from_tokens(self, sigmas_idx: Array, ctx_tokens: Array) -> Array:
        h = self._decode(sigmas_idx, ctx_tokens)
        logits_amp = self.amp_head(h)
        logits_phase = self.phase_head(h)
        if self.phase_mode == "softsign":
            logits_phase = jax.nn.soft_sign(logits_phase) * jnp.pi
        elif self.phase_mode != "raw":
            raise ValueError(f"Unknown phase_mode: {self.phase_mode}")
        logp = jax.nn.log_softmax(logits_amp, axis=-1)
        one_hot = jax.nn.one_hot(sigmas_idx, 4)
        log_amp = 0.5 * jnp.sum(jnp.sum(logp * one_hot, axis=-1), axis=-1)
        phase = jnp.sum(jnp.sum(logits_phase * one_hot, axis=-1), axis=-1)
        return log_amp + 1j * phase

    def __call__(self, h_fields: Array, t_idx: Array, sigmas_idx: Array) -> Array:
        bsz = h_fields.shape[0]
        t_idx = jnp.asarray(t_idx, dtype=jnp.int32)
        if t_idx.ndim == 0:
            t_idx = jnp.full((bsz,), t_idx, dtype=jnp.int32)
        m_traj, _ = self.encode_field(h_fields)
        m_t = m_traj[jnp.arange(bsz), t_idx]
        return self.log_psi_from_tokens(sigmas_idx, m_t)




In [5]:
@eqx.filter_jit
def sample_autoregressive_sigma(key, model, ctx, n_spins):
    bsz = ctx.shape[0]
    sigmas = jnp.zeros((bsz, n_spins), dtype=jnp.int32)

    if model.self_kv_cache:
        pos = model.pos_embed(jnp.arange(n_spins, dtype=jnp.int32))
        cross_kv_cache = model.build_cross_kv_cache(ctx)
        self_kv_cache = model.empty_self_kv_cache(bsz, n_spins)
        token_x0 = jnp.tile(model.start_token, (bsz, 1, 1)) + pos[None, 0:1, :]

        def body(carry, i):
            s, token_x, self_kv_cache, rng = carry
            x, self_kv_cache = model._decode_token_step(token_x, self_kv_cache, cross_kv_cache, i, n_spins)
            h = model.ln_final(x)
            logits = model.amp_head(h)[:, 0, :]
            rng, sub = random.split(rng)
            s_i = random.categorical(sub, logits).astype(jnp.int32)
            s = s.at[:, i].set(s_i)
            next_pos = jnp.take(pos, jnp.minimum(i + 1, n_spins - 1), axis=0)[None, None, :]
            next_token_x = model.token_embed(s_i)[:, None, :] + next_pos
            return (s, next_token_x, self_kv_cache, rng), None

        (sigmas, _, _, _), _ = lax.scan(body, (sigmas, token_x0, self_kv_cache, key), jnp.arange(n_spins, dtype=jnp.int32))
        return sigmas

    def body(carry, i):
        s, rng = carry
        logits = model.logits_from_tokens(s, ctx)[:, i]
        rng, sub = random.split(rng)
        s_i = random.categorical(sub, logits).astype(jnp.int32)
        s = s.at[:, i].set(s_i)
        return (s, rng), None

    (sigmas, _), _ = lax.scan(body, (sigmas, key), jnp.arange(n_spins))
    return sigmas


@eqx.filter_jit
def compute_local_energy_propagator(sigmas_idx, ctx, h_x_t, h_z_t, model, bond_i, bond_j, j_zz):
    alpha_idx = sigma_to_alpha_bits(sigmas_idx)
    alpha_pm = 1 - 2 * alpha_idx.astype(jnp.int32)
    log_u = model.log_psi_from_tokens(sigmas_idx, ctx)

    e_zz = -j_zz * jnp.sum(alpha_pm[:, bond_i] * alpha_pm[:, bond_j], axis=1)
    e_z = -h_z_t * jnp.sum(alpha_pm, axis=1)

    def ratio_fn(i):
        s_flip = sigma_flip_alpha(sigmas_idx, i)
        lp_flip = model.log_psi_from_tokens(s_flip, ctx)
        return jnp.exp(lp_flip - log_u)

    ratios = jax.vmap(ratio_fn)(jnp.arange(sigmas_idx.shape[1], dtype=jnp.int32))
    e_x = -h_x_t * jnp.sum(ratios, axis=0)
    return e_zz + e_z + e_x




def identity_anchor_loss(model: PropagatorNOQS, key: PRNGKey, n_samples: int, n_spins: int) -> Array:
    key_on, key_off = random.split(key)
    ctx = jnp.broadcast_to(model.get_initial_ctx(), (1, model.ctx_tokens, model.embed_dim))
    on_support = sample_identity_on_support(key_on, 1, n_samples, n_spins)
    ctx_tiled = jnp.broadcast_to(ctx[:, None, :, :], (1, n_samples, model.ctx_tokens, model.embed_dim)).reshape(
        n_samples, model.ctx_tokens, model.embed_dim
    )
    sigmas_on_flat = on_support.reshape(n_samples, n_spins)
    log_u_on = model.log_psi_from_tokens(sigmas_on_flat, ctx_tiled)

    target_log_prob = -float(n_spins) * jnp.log(2.0)
    pred_log_prob = 2.0 * jnp.real(log_u_on)
    l_amp = jnp.mean((pred_log_prob - target_log_prob) ** 2)

    pred_phase = jnp.imag(log_u_on)[None, :]
    phase_diff = pred_phase - lax.stop_gradient(jnp.mean(pred_phase, axis=1, keepdims=True))
    phase_wrapped = jnp.arctan2(jnp.sin(phase_diff), jnp.cos(phase_diff))
    l_phase = jnp.mean(phase_wrapped**2)

    off_support = sample_identity_off_support(key_off, on_support).reshape(n_samples, n_spins)
    log_u_off = model.log_psi_from_tokens(off_support, ctx_tiled)
    log_prob_off = 2.0 * jnp.real(log_u_off)
    l_off = jnp.mean(jnp.maximum(log_prob_off + 10.0, 0.0) ** 2)
    return l_amp + l_phase + l_off




def build_all_sigma_tokens(n_spins: int) -> Array:
    dim = 2**n_spins
    idx = jnp.arange(dim, dtype=jnp.uint32)
    bit_sites = jnp.arange(n_spins, dtype=jnp.uint32)
    bits = ((idx[:, None] & (jnp.uint32(1) << bit_sites[None, :])) > 0).astype(jnp.int32)
    alpha_grid = jnp.repeat(bits[:, None, :], dim, axis=1)
    beta_grid = jnp.repeat(bits[None, :, :], dim, axis=0)
    return (2 * alpha_grid + beta_grid).reshape(dim * dim, n_spins)


def build_all_z_bits(n_spins: int) -> Array:
    dim = 2**n_spins
    idx = jnp.arange(dim, dtype=jnp.uint32)
    bit_sites = jnp.arange(n_spins, dtype=jnp.uint32)
    return ((idx[:, None] & (jnp.uint32(1) << bit_sites[None, :])) > 0).astype(jnp.int32)


def sample_state_benchmark_specs(key: PRNGKey, count: int, n_spins: int, basis_family: str) -> Tuple[Array, Array]:
    key_bits, key_basis = random.split(key)
    bits = random.bernoulli(key_bits, 0.5, (count, n_spins)).astype(jnp.int32)
    if basis_family == "z":
        basis_ids = jnp.zeros((count,), dtype=jnp.int32)
    elif basis_family == "x":
        basis_ids = jnp.ones((count,), dtype=jnp.int32)
    elif basis_family == "both":
        basis_ids = random.bernoulli(key_basis, 0.5, (count,)).astype(jnp.int32)
    else:
        raise ValueError(f"Unknown state benchmark basis family: {basis_family}")
    return basis_ids, bits


def product_state_to_z_amplitudes(bits: Array, basis: str, all_z_bits: Array) -> Array:
    dim = all_z_bits.shape[0]
    if basis == "z":
        basis_index = int(sum((int(bits[i]) << i) for i in range(bits.shape[0])))
        return jnp.zeros((dim,), dtype=jnp.complex64).at[basis_index].set(1.0 + 0.0j)
    if basis == "x":
        parity = jnp.mod(jnp.sum(all_z_bits * bits[None, :], axis=1), 2)
        signs = 1.0 - 2.0 * parity.astype(jnp.float32)
        amps = signs / jnp.sqrt(float(dim))
        return amps.astype(jnp.complex64)
    raise ValueError(f"Unknown product-state basis: {basis}")


def parse_state_bits(bits_spec: str, n_spins: int) -> Array:
    if bits_spec is None:
        raise ValueError("state benchmark fixed bits must be provided")
    cleaned = bits_spec.replace(" ", "").replace(",", "")
    if len(cleaned) != n_spins or any(ch not in "01" for ch in cleaned):
        raise ValueError(f"Expected a {n_spins}-bit 0/1 string, got: {bits_spec}")
    return jnp.asarray([int(ch) for ch in cleaned], dtype=jnp.int32)


def run_single_state_fidelity_benchmark(
    current_model,
    h_fields_1d,
    *,
    n_spins: int,
    t_steps: int,
    t_max: float,
    dt: float,
    apply_H_exact,
    basis: str,
    bits: Array,
    alpha_chunk: int,
    beta_chunk: int,
    max_x_spins: int,
):
    all_z_bits = build_all_z_bits(n_spins)
    if basis == "x" and n_spins > max_x_spins:
        return {
            "summary": {
                "num_requested": 1,
                "num_evaluated": 0,
                "num_skipped": 1,
                "basis_family": basis,
                "avg_of_avg_fidelity": None,
                "avg_of_final_fidelity": None,
                "best_final_fidelity": None,
                "worst_final_fidelity": None,
                "t_max": float(t_max),
                "t_steps": int(t_steps),
            },
            "states": [{
                "state_index": 0,
                "basis": basis,
                "bits": [int(x) for x in np.asarray(bits)],
                "skipped": True,
                "reason": f"X-basis benchmark requires summing over all Z-basis columns and is disabled for n_spins > {max_x_spins}.",
            }],
        }

    m_traj, _ = current_model.encode_field(h_fields_1d[None, :, :])
    m_traj = m_traj[0]
    psi_exact = product_state_to_z_amplitudes(bits, basis, all_z_bits)
    coeffs_beta = psi_exact if basis == "x" else None
    fids = []

    for t_idx in range(t_steps):
        m_i = m_traj[t_idx]
        if basis == "z":
            psi_pred = apply_propagator_to_z_state(current_model, m_i, bits, all_z_bits, alpha_chunk)
        else:
            psi_pred = apply_propagator_to_x_state(current_model, m_i, coeffs_beta, all_z_bits, alpha_chunk, beta_chunk)

        ov = jnp.vdot(psi_exact, psi_pred)
        fids.append(float(jnp.abs(ov) ** 2))

        if t_idx < t_steps - 1:
            h_x_curr = h_fields_1d[t_idx, 0]
            h_z_curr = h_fields_1d[t_idx, 1]
            h_x_next = h_fields_1d[t_idx + 1, 0]
            h_z_next = h_fields_1d[t_idx + 1, 1]
            h_x_mid = 0.5 * (h_x_curr + h_x_next)
            h_z_mid = 0.5 * (h_z_curr + h_z_next)
            k1 = -1j * apply_H_exact(psi_exact, h_x_curr, h_z_curr)
            k2 = -1j * apply_H_exact(psi_exact + 0.5 * dt * k1, h_x_mid, h_z_mid)
            k3 = -1j * apply_H_exact(psi_exact + 0.5 * dt * k2, h_x_mid, h_z_mid)
            k4 = -1j * apply_H_exact(psi_exact + dt * k3, h_x_next, h_z_next)
            psi_exact = psi_exact + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
            psi_exact = psi_exact / jnp.clip(jnp.linalg.norm(psi_exact), a_min=1e-12)

    record = {
        "state_index": 0,
        "basis": basis,
        "bits": [int(x) for x in np.asarray(bits)],
        "avg_fidelity": float(np.mean(fids)),
        "final_fidelity": float(fids[-1]),
        "best_fidelity": float(np.max(fids)),
        "worst_fidelity": float(np.min(fids)),
        "fidelity_series": fids,
    }
    summary = {
        "num_requested": 1,
        "num_evaluated": 1,
        "num_skipped": 0,
        "basis_family": basis,
        "avg_of_avg_fidelity": record["avg_fidelity"],
        "avg_of_final_fidelity": record["final_fidelity"],
        "best_final_fidelity": record["final_fidelity"],
        "worst_final_fidelity": record["final_fidelity"],
        "t_max": float(t_max),
        "t_steps": int(t_steps),
    }
    return {"summary": summary, "states": [record]}


def apply_propagator_to_z_state(current_model, m_i: Array, beta_bits: Array, all_z_bits: Array, alpha_chunk: int) -> Array:
    pieces = []
    beta_bits = jnp.asarray(beta_bits, dtype=jnp.int32)
    dim = all_z_bits.shape[0]
    for start in range(0, dim, alpha_chunk):
        stop = min(start + alpha_chunk, dim)
        alpha_chunk_bits = all_z_bits[start:stop]
        sigma_chunk = 2 * alpha_chunk_bits + beta_bits[None, :]
        m_batch = jnp.broadcast_to(m_i[None, :, :], (sigma_chunk.shape[0], m_i.shape[0], m_i.shape[1]))
        log_u = current_model.log_psi_from_tokens(sigma_chunk, m_batch)
        pieces.append(jnp.exp(log_u))
    psi = jnp.concatenate(pieces, axis=0)
    return psi / jnp.clip(jnp.linalg.norm(psi), a_min=1e-12)


def apply_propagator_to_z_states(
    current_model,
    m_i: Array,
    beta_bits_batch: Array,
    all_z_bits: Array,
    alpha_chunk: int,
) -> Array:
    pieces = []
    beta_bits_batch = jnp.asarray(beta_bits_batch, dtype=jnp.int32)
    n_states = beta_bits_batch.shape[0]
    dim = all_z_bits.shape[0]
    for start in range(0, dim, alpha_chunk):
        stop = min(start + alpha_chunk, dim)
        alpha_chunk_bits = all_z_bits[start:stop]
        sigma_chunk = 2 * alpha_chunk_bits[:, None, :] + beta_bits_batch[None, :, :]
        sigma_chunk = sigma_chunk.reshape(-1, alpha_chunk_bits.shape[-1])
        m_batch = jnp.broadcast_to(m_i[None, :, :], (sigma_chunk.shape[0], m_i.shape[0], m_i.shape[1]))
        log_u = current_model.log_psi_from_tokens(sigma_chunk, m_batch)
        pieces.append(jnp.exp(log_u).reshape(stop - start, n_states))
    psi = jnp.concatenate(pieces, axis=0)
    return psi / jnp.clip(jnp.linalg.norm(psi, axis=0, keepdims=True), a_min=1e-12)


def apply_propagator_to_x_state(
    current_model,
    m_i: Array,
    coeffs_beta: Array,
    all_z_bits: Array,
    alpha_chunk: int,
    beta_chunk: int,
) -> Array:
    pieces = []
    dim = all_z_bits.shape[0]
    for alpha_start in range(0, dim, alpha_chunk):
        alpha_stop = min(alpha_start + alpha_chunk, dim)
        alpha_chunk_bits = all_z_bits[alpha_start:alpha_stop]
        accum = jnp.zeros((alpha_stop - alpha_start,), dtype=jnp.complex64)
        for beta_start in range(0, dim, beta_chunk):
            beta_stop = min(beta_start + beta_chunk, dim)
            beta_chunk_bits = all_z_bits[beta_start:beta_stop]
            sigma_chunk = 2 * alpha_chunk_bits[:, None, :] + beta_chunk_bits[None, :, :]
            sigma_chunk = sigma_chunk.reshape(-1, alpha_chunk_bits.shape[-1])
            m_batch = jnp.broadcast_to(m_i[None, :, :], (sigma_chunk.shape[0], m_i.shape[0], m_i.shape[1]))
            log_u = current_model.log_psi_from_tokens(sigma_chunk, m_batch)
            u_block = jnp.exp(log_u).reshape(alpha_stop - alpha_start, beta_stop - beta_start)
            accum = accum + u_block @ coeffs_beta[beta_start:beta_stop]
        pieces.append(accum)
    psi = jnp.concatenate(pieces, axis=0)
    return psi / jnp.clip(jnp.linalg.norm(psi), a_min=1e-12)


def run_state_fidelity_benchmark_z_vectorized(
    current_model,
    h_fields_1d,
    *,
    n_spins: int,
    t_steps: int,
    dt: float,
    apply_H_exact,
    bitstrings: Array,
    alpha_chunk: int,
    beta_chunk: int,
):
    all_z_bits = build_all_z_bits(n_spins)
    m_traj, _ = current_model.encode_field(h_fields_1d[None, :, :])
    m_traj = m_traj[0]

    n_states = int(bitstrings.shape[0])
    dim = all_z_bits.shape[0]
    idx = jnp.sum(bitstrings.astype(jnp.uint32) * (jnp.uint32(1) << jnp.arange(n_spins, dtype=jnp.uint32)[None, :]), axis=1)
    psi_exact = jnp.eye(dim, dtype=jnp.complex64)[:, idx]
    fidelity_series_np = []

    def apply_h_cols(mat, h_x_val, h_z_val):
        return jax.vmap(lambda col: apply_H_exact(col, h_x_val, h_z_val), in_axes=1, out_axes=1)(mat)

    @jax.jit
    def rk4_step(psi_exact, h_x_curr, h_z_curr, h_x_next, h_z_next):
        h_x_mid = 0.5 * (h_x_curr + h_x_next)
        h_z_mid = 0.5 * (h_z_curr + h_z_next)
        k1 = -1j * apply_h_cols(psi_exact, h_x_curr, h_z_curr)
        k2 = -1j * apply_h_cols(psi_exact + 0.5 * dt * k1, h_x_mid, h_z_mid)
        k3 = -1j * apply_h_cols(psi_exact + 0.5 * dt * k2, h_x_mid, h_z_mid)
        k4 = -1j * apply_h_cols(psi_exact + dt * k3, h_x_next, h_z_next)
        psi_next = psi_exact + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
        return psi_next / jnp.clip(jnp.linalg.norm(psi_next, axis=0, keepdims=True), a_min=1e-12)

    for t_idx in range(t_steps):
        m_i = m_traj[t_idx]
        pred_parts = []
        for beta_start in range(0, n_states, beta_chunk):
            beta_stop = min(beta_start + beta_chunk, n_states)
            pred_parts.append(apply_propagator_to_z_states(current_model, m_i, bitstrings[beta_start:beta_stop], all_z_bits, alpha_chunk))
        psi_pred = jnp.concatenate(pred_parts, axis=1)
        overlaps = jnp.sum(jnp.conj(psi_exact) * psi_pred, axis=0)
        fidelity_series_np.append(np.asarray(jax.device_get(jnp.abs(overlaps) ** 2), dtype=np.float32))

        if t_idx < t_steps - 1:
            h_x_curr = h_fields_1d[t_idx, 0]
            h_z_curr = h_fields_1d[t_idx, 1]
            h_x_next = h_fields_1d[t_idx + 1, 0]
            h_z_next = h_fields_1d[t_idx + 1, 1]
            psi_exact = rk4_step(psi_exact, h_x_curr, h_z_curr, h_x_next, h_z_next)

    fidelity_series_np = np.stack(fidelity_series_np, axis=0)
    results = []
    avg_fids = []
    final_fids = []
    bitstrings_np = np.asarray(bitstrings)
    for state_idx in range(n_states):
        fids = fidelity_series_np[:, state_idx]
        record = {
            "state_index": int(state_idx),
            "basis": "z",
            "bits": [int(x) for x in bitstrings_np[state_idx]],
            "avg_fidelity": float(np.mean(fids)),
            "final_fidelity": float(fids[-1]),
            "best_fidelity": float(np.max(fids)),
            "worst_fidelity": float(np.min(fids)),
            "fidelity_series": [float(x) for x in fids],
        }
        results.append(record)
        avg_fids.append(record["avg_fidelity"])
        final_fids.append(record["final_fidelity"])

    summary = {
        "num_requested": int(n_states),
        "num_evaluated": int(n_states),
        "num_skipped": 0,
        "basis_family": "z",
        "avg_of_avg_fidelity": float(np.mean(avg_fids)),
        "avg_of_final_fidelity": float(np.mean(final_fids)),
        "best_final_fidelity": float(np.max(final_fids)),
        "worst_final_fidelity": float(np.min(final_fids)),
        "t_max": float(dt * (t_steps - 1)),
        "t_steps": int(t_steps),
    }
    return {"summary": summary, "states": results}


def run_state_fidelity_benchmark(
    current_model,
    h_fields_1d,
    *,
    n_spins: int,
    t_steps: int,
    t_max: float,
    dt: float,
    apply_H_exact,
    state_count: int,
    basis_family: str,
    sample_seed: int,
    alpha_chunk: int,
    beta_chunk: int,
    max_x_spins: int,
):
    if state_count <= 0:
        return None

    all_z_bits = build_all_z_bits(n_spins)
    basis_ids, bitstrings = sample_state_benchmark_specs(random.PRNGKey(sample_seed), state_count, n_spins, basis_family)
    if basis_family == "z":
        return run_state_fidelity_benchmark_z_vectorized(
            current_model,
            h_fields_1d,
            n_spins=n_spins,
            t_steps=t_steps,
            dt=dt,
            apply_H_exact=apply_H_exact,
            bitstrings=bitstrings,
            alpha_chunk=alpha_chunk,
            beta_chunk=beta_chunk,
        )

    m_traj, _ = current_model.encode_field(h_fields_1d[None, :, :])
    m_traj = m_traj[0]

    results = []
    avg_fids = []
    final_fids = []

    for state_idx in range(state_count):
        basis = "x" if int(basis_ids[state_idx]) == 1 else "z"
        bits = bitstrings[state_idx]
        record = {
            "state_index": int(state_idx),
            "basis": basis,
            "bits": [int(x) for x in np.asarray(bits)],
        }

        if basis == "x" and n_spins > max_x_spins:
            record["skipped"] = True
            record["reason"] = (
                f"X-basis benchmark requires summing over all Z-basis columns and is disabled for n_spins > {max_x_spins}."
            )
            results.append(record)
            continue

        psi_exact = product_state_to_z_amplitudes(bits, basis, all_z_bits)
        coeffs_beta = psi_exact if basis == "x" else None
        fids = []

        for t_idx in range(t_steps):
            m_i = m_traj[t_idx]
            if basis == "z":
                psi_pred = apply_propagator_to_z_state(current_model, m_i, bits, all_z_bits, alpha_chunk)
            else:
                psi_pred = apply_propagator_to_x_state(current_model, m_i, coeffs_beta, all_z_bits, alpha_chunk, beta_chunk)

            ov = jnp.vdot(psi_exact, psi_pred)
            fids.append(float(jnp.abs(ov) ** 2))

            if t_idx < t_steps - 1:
                h_x_curr = h_fields_1d[t_idx, 0]
                h_z_curr = h_fields_1d[t_idx, 1]
                h_x_next = h_fields_1d[t_idx + 1, 0]
                h_z_next = h_fields_1d[t_idx + 1, 1]
                h_x_mid = 0.5 * (h_x_curr + h_x_next)
                h_z_mid = 0.5 * (h_z_curr + h_z_next)
                k1 = -1j * apply_H_exact(psi_exact, h_x_curr, h_z_curr)
                k2 = -1j * apply_H_exact(psi_exact + 0.5 * dt * k1, h_x_mid, h_z_mid)
                k3 = -1j * apply_H_exact(psi_exact + 0.5 * dt * k2, h_x_mid, h_z_mid)
                k4 = -1j * apply_H_exact(psi_exact + dt * k3, h_x_next, h_z_next)
                psi_exact = psi_exact + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
                psi_exact = psi_exact / jnp.clip(jnp.linalg.norm(psi_exact), a_min=1e-12)

        record.update(
            avg_fidelity=float(np.mean(fids)),
            final_fidelity=float(fids[-1]),
            best_fidelity=float(np.max(fids)),
            worst_fidelity=float(np.min(fids)),
            fidelity_series=fids,
        )
        results.append(record)
        avg_fids.append(record["avg_fidelity"])
        final_fids.append(record["final_fidelity"])

    summary = {
        "num_requested": int(state_count),
        "num_evaluated": int(len(avg_fids)),
        "num_skipped": int(sum(1 for r in results if r.get("skipped"))),
        "basis_family": basis_family,
        "avg_of_avg_fidelity": None if not avg_fids else float(np.mean(avg_fids)),
        "avg_of_final_fidelity": None if not final_fids else float(np.mean(final_fids)),
        "best_final_fidelity": None if not final_fids else float(np.max(final_fids)),
        "worst_final_fidelity": None if not final_fids else float(np.min(final_fids)),
        "t_max": float(t_max),
        "t_steps": int(t_steps),
    }
    return {"summary": summary, "states": results}


def run_propagator_benchmark(current_model, h_fields_1d, *, n_spins, t_steps, t_max, dt, apply_H_exact):
    dim = 2**n_spins
    sigmas_all = build_all_sigma_tokens(n_spins)
    m_traj, _ = current_model.encode_field(h_fields_1d[None, :, :])
    m_traj = m_traj[0]
    u_exact = jnp.eye(dim, dtype=jnp.complex64) / jnp.sqrt(float(dim))
    operator_fids = []
    column_fids = []
    fro_errors = []

    def apply_h_matrix(mat, h_x_val, h_z_val):
        return jax.vmap(lambda col: apply_H_exact(col, h_x_val, h_z_val), in_axes=1, out_axes=1)(mat)

    for i in range(t_steps):
        m_i = m_traj[i]
        m_batch = jnp.broadcast_to(m_i[None, :, :], (sigmas_all.shape[0], m_i.shape[0], m_i.shape[1]))
        log_u = current_model.log_psi_from_tokens(sigmas_all, m_batch)
        u_pred = jnp.exp(log_u).reshape(dim, dim)

        ov = jnp.vdot(u_exact.reshape(-1), u_pred.reshape(-1))
        operator_fids.append(float(jnp.abs(ov) ** 2))
        phase = jnp.where(jnp.abs(ov) > 1e-12, jnp.conj(ov) / jnp.abs(ov), jnp.asarray(1.0 + 0.0j, dtype=ov.dtype))
        fro_errors.append(float(jnp.linalg.norm(u_exact - phase * u_pred)))

        exact_norms = jnp.clip(jnp.linalg.norm(u_exact, axis=0), a_min=1e-12)
        pred_norms = jnp.clip(jnp.linalg.norm(u_pred, axis=0), a_min=1e-12)
        exact_cols = u_exact / exact_norms[None, :]
        pred_cols = u_pred / pred_norms[None, :]
        col_ovs = jnp.sum(jnp.conj(exact_cols) * pred_cols, axis=0)
        column_fids.append(float(jnp.mean(jnp.abs(col_ovs) ** 2)))

        if i < t_steps - 1:
            h_x_curr = h_fields_1d[i, 0]
            h_z_curr = h_fields_1d[i, 1]
            h_x_next = h_fields_1d[i + 1, 0]
            h_z_next = h_fields_1d[i + 1, 1]
            h_x_mid = 0.5 * (h_x_curr + h_x_next)
            h_z_mid = 0.5 * (h_z_curr + h_z_next)
            k1 = -1j * apply_h_matrix(u_exact, h_x_curr, h_z_curr)
            k2 = -1j * apply_h_matrix(u_exact + 0.5 * dt * k1, h_x_mid, h_z_mid)
            k3 = -1j * apply_h_matrix(u_exact + 0.5 * dt * k2, h_x_mid, h_z_mid)
            k4 = -1j * apply_h_matrix(u_exact + dt * k3, h_x_next, h_z_next)
            u_exact = u_exact + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
            u_exact = u_exact / jnp.clip(jnp.linalg.norm(u_exact), a_min=1e-12)

    return {
        "summary": {
            "avg_operator_fidelity": float(np.mean(operator_fids)),
            "final_operator_fidelity": float(operator_fids[-1]),
            "best_operator_fidelity": float(np.max(operator_fids)),
            "avg_column_fidelity": float(np.mean(column_fids)),
            "final_column_fidelity": float(column_fids[-1]),
            "best_column_fidelity": float(np.max(column_fids)),
            "final_fro_error": float(fro_errors[-1]),
            "avg_fro_error": float(np.mean(fro_errors)),
        },
        "series": {
            "operator_fidelity": operator_fids,
            "column_fidelity": column_fids,
            "fro_error": fro_errors,
        },
    }




In [7]:
parser = build_parser()
args = parser.parse_args(["--Lx", "4", "--Ly", "4", "--checkpoint-path", "learn_propagator_latest_checkpoint_4x4.pkl",
                          #"--batch-size", "4", "--num-time-points", "3",
                          #"--num-layers", "4", "--ctx-tokens", "8",
                          "--decay-steps", "2000",
                          #"--t-max", "0.5", "--start-lr", "5e-4",
                          "--anchor-weight", "1.0"])
                          #"--time-curriculum",              # not set before
                          #"--time-curriculum-start-frac", "0.3", ])
if not (0.0 < float(args.time_curriculum_start_frac) <= 1.0):
    raise ValueError("time_curriculum_start_frac must be in (0, 1]")
if float(args.time_curriculum_power) <= 0.0:
    raise ValueError("time_curriculum_power must be > 0")
print(f"JAX is using: {jax.devices()}")

output_dir = Path(args.output_dir).expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = Path(args.checkpoint_path).expanduser().resolve()
resume_path = Path(args.resume).expanduser().resolve() if args.resume is not None else None

n_spins = int(args.Lx * args.Ly)
print(f"System: {args.Lx}x{args.Ly} = {n_spins} spins")
bonds_i_np, bonds_j_np = make_lattice_bonds(args.Lx, args.Ly, args.site_order)
bond_i = jnp.asarray(bonds_i_np, dtype=jnp.int32)
bond_j = jnp.asarray(bonds_j_np, dtype=jnp.int32)
j_zz = float(J_zz)
apply_H_exact = make_apply_H_exact(bonds_i_np, bonds_j_np, n_spins, j_zz=j_zz)
fno_modes = int(FNO_MODES)
if args.canonical_M0:
    fixed_M0_mode = "canonical_basis"
elif args.diff_M:
    fixed_M0_mode = None
else:
    fixed_M0_mode = "random_unit_rows"

dt = float(args.t_max) / float(args.t_steps - 1)

print("--- Training Propagator NOQS Model (TFIM + longitudinal field, OBC) ---")
if fixed_M0_mode is None:
    print("M0 mode: shared learnable initial context")
else:
    print(f"M0 mode: {fixed_M0_mode}")
print(f"Phase mode: {args.phase_mode}")
print(f"Gauge centering: {args.gauge_centering}")
print(f"Score function weight: {args.score_function_weight}")
print(f"Self-KV cache: {args.self_kv_cache}")
print(f"Context dynamics: direct FNO Mdot with {args.integration_rule} integration")
print(f"Site order: {args.site_order}")
if args.time_curriculum:
    print(
        f"Time curriculum: enabled, sampled-time window starts at {args.time_curriculum_start_frac:.2f} "
        f"of the grid with power {args.time_curriculum_power:.2f}"
    )
else:
    print("Time curriculum: disabled")

rng = random.PRNGKey(args.seed)
model_key, rng = random.split(rng)
model = PropagatorNOQS(
    args.d_model,
    8,
    args.num_layers,
    args.mlp_mult,
    fno_modes,
    args.fno_width,
    args.ctx_tokens,
    2,
    phase_mode=args.phase_mode,
    self_kv_cache=args.self_kv_cache,
    fixed_M0_mode=fixed_M0_mode,
    fixed_M0_seed=args.seed if fixed_M0_mode == "random_unit_rows" else None,
    n_spins=n_spins,
    integration_rule=args.integration_rule,
    dt=dt,
    key=model_key,
)
model_template = model

schedule = optax.exponential_decay(
    float(args.start_lr),
    int(args.decay_steps),
    float(args.decay_rate),
    end_value=float(args.min_lr),
)
optimizer = optax.chain(optax.clip_by_global_norm(0.1), optax.adam(learning_rate=schedule))
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))
start_step = 0
loss_hist = []
best_loss = float("inf")
best_model = model

checkpoint_config = {
    "fixed_M0_mode": fixed_M0_mode,
    "score_function_weight": float(args.score_function_weight),
    "phase_mode": args.phase_mode,
    "gauge_centering": args.gauge_centering,
    "site_order": args.site_order,
    "self_kv_cache": bool(args.self_kv_cache),
    "context_dynamics": "direct_fno_mdot_propagator",
    "integration_rule": args.integration_rule,
    "mlp_mult": int(args.mlp_mult),
    "START_LR": float(args.start_lr),
    "MIN_LR": float(args.min_lr),
    "DECAY_RATE": float(args.decay_rate),
    "DECAY_STEPS": int(args.decay_steps),
    "T_MAX": float(args.t_max),
    "T_STEPS": int(args.t_steps),
    "BATCH_SIZE": int(args.batch_size),
    "K": int(args.num_time_points),
    "M_SAMPLES": int(args.num_mc_samples),
    "time_curriculum": bool(args.time_curriculum),
    "time_curriculum_start_frac": float(args.time_curriculum_start_frac),
    "time_curriculum_power": float(args.time_curriculum_power),
}

if resume_path is not None:
    if not resume_path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")
    resume_payload = load_resume_payload(resume_path)
    if isinstance(resume_payload, dict) and resume_payload.get("format") == "learn_propagator_checkpoint_v1":
        saved_cfg = resume_payload.get("config", {})
        saved_dt = getattr(resume_payload["model"], "dt", None)
        if saved_dt is not None and abs(float(saved_dt) - float(dt)) > 1e-12:
            tqdm.write(
                "Resuming with updated time discretization: "
                f"saved_dt={float(saved_dt):.15g}, current_dt={float(dt):.15g}, "
                f"saved_T_MAX={saved_cfg.get('T_MAX', 'unknown')}, "
                f"saved_T_STEPS={saved_cfg.get('T_STEPS', 'unknown')}, "
                f"current_T_MAX={float(args.t_max)}, current_T_STEPS={int(args.t_steps)}"
            )
        saved_model = upgrade_resume_model(resume_payload["model"], model_template)
        saved_best_model = upgrade_resume_model(resume_payload.get("best_model", resume_payload["model"]), model_template)
        model = combine_model(saved_model, model_template)
        best_model = combine_model(saved_best_model, model_template)
        opt_state = upgrade_resume_model_in_tree(resume_payload["opt_state"], model_template)
        rng = jnp.asarray(resume_payload["rng"], dtype=jnp.uint32)
        best_loss = float(resume_payload.get("best_loss", float("inf")))
        loss_hist = [float(x) for x in resume_payload.get("loss_hist", [])]
        start_step = int(resume_payload.get("step", -1)) + 1
        print(f"Loaded resumable checkpoint from: {resume_path}")
        print(f"Resuming TDVP training from step {start_step}")
    else:
        model = combine_model(resume_payload, model_template)
        best_model = model
        opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))
        print(f"Loaded params-only checkpoint from: {resume_path}")
        print("Optimizer state was reinitialized; starting TDVP training from step 0")

@eqx.filter_jit
def anchor_step(current_model, current_opt_state, key):
    loss, grads = eqx.filter_value_and_grad(identity_anchor_loss)(current_model, key, args.batch_size, n_spins)
    updates, current_opt_state = optimizer.update(grads, current_opt_state, eqx.filter(current_model, eqx.is_inexact_array))
    current_model = eqx.apply_updates(current_model, updates)
    return current_model, current_opt_state, loss

@eqx.filter_jit
def noqs_loss(current_model, key, h_fields, t_idx, anchor_w):
    batch_size = h_fields.shape[0]
    k_s, k_a = random.split(key)
    m_traj, mdot_traj = current_model.encode_field(h_fields)
    batch_idx = jnp.arange(batch_size)[:, None]
    m_t = m_traj[batch_idx, t_idx]
    mdot_t = mdot_traj[batch_idx, t_idx]
    h_t = h_fields[batch_idx, t_idx, :]
    h_x_t = h_t[..., 0]
    h_z_t = h_t[..., 1]

    num_groups = batch_size * t_idx.shape[1]
    m_groups = m_t.reshape(num_groups, args.ctx_tokens, args.d_model)
    mdot_groups = mdot_t.reshape(num_groups, args.ctx_tokens, args.d_model)
    h_x_groups = h_x_t.reshape(num_groups)
    h_z_groups = h_z_t.reshape(num_groups)

    m_flat = jnp.broadcast_to(
        m_groups[:, None, :, :],
        (num_groups, args.num_mc_samples, args.ctx_tokens, args.d_model),
    ).reshape(-1, args.ctx_tokens, args.d_model)
    mdot_flat = jnp.broadcast_to(
        mdot_groups[:, None, :, :],
        (num_groups, args.num_mc_samples, args.ctx_tokens, args.d_model),
    ).reshape(-1, args.ctx_tokens, args.d_model)
    h_x_flat = jnp.broadcast_to(h_x_groups[:, None], (num_groups, args.num_mc_samples)).reshape(-1)
    h_z_flat = jnp.broadcast_to(h_z_groups[:, None], (num_groups, args.num_mc_samples)).reshape(-1)

    sigmas_idx = sample_autoregressive_sigma(k_s, current_model, m_flat, n_spins)

    def logu_and_dt_logu(m, mdot, s):
        def _f(x):
            return current_model.log_psi_from_tokens(s[None], x[None])[0]
        return jax.jvp(_f, (m,), (mdot,))

    if args.score_function_weight == 0.0:
        def dt_only(m, mdot, s):
            return logu_and_dt_logu(m, mdot, s)[1]
        dt_logu = jax.vmap(dt_only)(m_flat, mdot_flat, sigmas_idx)
        log_u_base = None
    else:
        log_u_base, dt_logu = jax.vmap(logu_and_dt_logu)(m_flat, mdot_flat, sigmas_idx)

    e_loc = compute_local_energy_propagator(sigmas_idx, m_flat, h_x_flat, h_z_flat, current_model, bond_i, bond_j, j_zz)
    diff = 1j * dt_logu - e_loc
    if args.gauge_centering == "global":
        diff_centered = diff - lax.stop_gradient(jnp.mean(diff))
        sample_loss = jnp.abs(diff_centered) ** 2
        loss_phys = jnp.mean(sample_loss)
        score_weight = sample_loss - loss_phys
    elif args.gauge_centering == "per_time":
        diff_grouped = diff.reshape(num_groups, args.num_mc_samples)
        diff_centered = diff_grouped - lax.stop_gradient(jnp.mean(diff_grouped, axis=1, keepdims=True))
        sample_loss = jnp.abs(diff_centered) ** 2
        loss_phys = jnp.mean(sample_loss)
        score_weight = sample_loss.reshape(-1) - loss_phys
    else:
        raise ValueError(f"Unknown gauge centering: {args.gauge_centering}")

    l_anchor = identity_anchor_loss(current_model, k_a, args.batch_size, n_spins)
    physical_loss = loss_phys + anchor_w * l_anchor
    if args.score_function_weight == 0.0:
        opt_loss = physical_loss
    else:
        log_p = 2.0 * jnp.real(log_u_base)
        loss_score = jnp.mean(lax.stop_gradient(score_weight) * log_p)
        opt_loss = physical_loss + args.score_function_weight * loss_score
    return opt_loss, physical_loss

@eqx.filter_jit
def train_step(current_model, current_opt_state, key, max_t_idx):
    k_fields, k_times, k_loss = random.split(key, 3)
    h_batch = generate_grf_trajectories(k_fields, args.batch_size, args.t_steps, args.t_max)
    t_idx = random.randint(k_times, (args.batch_size, args.num_time_points), 0, max_t_idx)
    (loss, physical_loss), grads = eqx.filter_value_and_grad(noqs_loss, has_aux=True)(
        current_model, k_loss, h_batch, t_idx, args.anchor_weight
    )
    updates, current_opt_state = optimizer.update(grads, current_opt_state, eqx.filter(current_model, eqx.is_inexact_array))
    current_model = eqx.apply_updates(current_model, updates)
    return current_model, current_opt_state, physical_loss

if resume_path is None:
    print("Pre-training identity anchor...")
    for i in tqdm(range(args.anchor_pretrain_steps)):
        rng, k = random.split(rng)
        model, opt_state, _ = anchor_step(model, opt_state, k)
else:
    print("Skipping anchor pre-training because --resume was provided.")

print("TDVP Training...")
benchmark_history = []
pbar = tqdm(range(start_step, args.steps), initial=start_step, total=args.steps)
last_completed_step = start_step - 1

for step in pbar:
    rng, k_step = random.split(rng)
    current_max_t_idx = sampled_time_curriculum_max_t(args, step)
    model, opt_state, loss = train_step(
        model,
        opt_state,
        k_step,
        jnp.asarray(current_max_t_idx, dtype=jnp.int32),
    )
    last_completed_step = step

    current_loss = float(loss)
    loss_hist.append(current_loss)
    if current_loss < best_loss:
        best_loss = current_loss
        best_model = model

    if step % args.print_every == 0:
        if args.time_curriculum:
            curriculum_frac = sampled_time_curriculum_fraction(args, step)
            pbar.set_description(
                f"Loss: {np.mean(loss_hist[-30:]):.5f}, "
                f"t_frac: {curriculum_frac:.2f}, t_max_idx: {current_max_t_idx - 1}"
            )
        else:
            pbar.set_description(f"Loss: {np.mean(loss_hist[-30:]):.5f}")

    if step > 0 and step % 500 == 0:
        save_training_checkpoint(
            checkpoint_path,
            step=step,
            model=model,
            opt_state=opt_state,
            rng=rng,
            best_loss=best_loss,
            best_model=best_model,
            loss_hist=loss_hist,
            config=checkpoint_config,
        )

    if args.benchmark_every > 0 and n_spins <= 4 and (step + 1) % args.benchmark_every == 0:
        h_val = generate_grf_trajectories(random.PRNGKey(args.benchmark_key), 1, args.t_steps, args.t_max)[0]
        benchmark_result = run_propagator_benchmark(
            model,
            h_val,
            n_spins=n_spins,
            t_steps=args.t_steps,
            t_max=args.t_max,
            dt=dt,
            apply_H_exact=apply_H_exact,
        )
        benchmark_record = {"step": int(step), **benchmark_result["summary"]}
        benchmark_history.append(benchmark_record)
        tqdm.write(
            f"[benchmark step {step}] "
            f"avg_op_fid={benchmark_record['avg_operator_fidelity']:.6f}, "
            f"final_op_fid={benchmark_record['final_operator_fidelity']:.6f}, "
            f"avg_col_fid={benchmark_record['avg_column_fidelity']:.6f}, "
            f"final_col_fid={benchmark_record['final_column_fidelity']:.6f}, "
            f"final_fro_err={benchmark_record['final_fro_error']:.6e}"
        )
        save_benchmark_series(output_dir, step, benchmark_result)
        save_json(output_dir / "benchmark_history.json", {"benchmarks": benchmark_history})
        save_json(output_dir / "benchmark_summary.json", benchmark_result["summary"])
        save_training_checkpoint(
            checkpoint_path,
            step=step,
            model=model,
            opt_state=opt_state,
            rng=rng,
            best_loss=best_loss,
            best_model=best_model,
            loss_hist=loss_hist,
            config=checkpoint_config,
        )

print(f"Training Complete. Best loss achieved: {best_loss:.6f}")

save_training_checkpoint(
    checkpoint_path,
    step=last_completed_step,
    model=model,
    opt_state=opt_state,
    rng=rng,
    best_loss=best_loss,
    best_model=best_model,
    loss_hist=loss_hist,
    config=checkpoint_config,
)

benchmark = None
if args.benchmark_every > 0 and n_spins <= 4:
    benchmark_history_path = output_dir / "benchmark_history.json"
    if benchmark_history_path.exists():
        benchmark_payload = json.loads(benchmark_history_path.read_text())
        if benchmark_payload.get("benchmarks"):
            benchmark = benchmark_payload["benchmarks"][-1]

state_benchmark = None
if args.state_benchmark_fixed_basis is not None:
    bits = parse_state_bits(args.state_benchmark_fixed_bits, n_spins)
    h_state = generate_grf_trajectories(random.PRNGKey(args.benchmark_key), 1, args.t_steps, args.t_max)[0]
    state_benchmark = run_single_state_fidelity_benchmark(
        model,
        h_state,
        n_spins=n_spins,
        t_steps=args.t_steps,
        t_max=args.t_max,
        dt=dt,
        apply_H_exact=apply_H_exact,
        basis=args.state_benchmark_fixed_basis,
        bits=bits,
        alpha_chunk=args.state_benchmark_alpha_chunk,
        beta_chunk=args.state_benchmark_beta_chunk,
        max_x_spins=args.state_benchmark_max_x_spins,
    )
    save_json(output_dir / "state_benchmark.json", state_benchmark)
    sb_summary = state_benchmark["summary"]
    print(
        "[state benchmark] "
        f"evaluated={sb_summary['num_evaluated']}, "
        f"skipped={sb_summary['num_skipped']}, "
        f"avg_final_fid={sb_summary['avg_of_final_fidelity']}"
    )
elif args.state_benchmark_count > 0:
    h_state = generate_grf_trajectories(random.PRNGKey(args.benchmark_key), 1, args.t_steps, args.t_max)[0]
    state_benchmark = run_state_fidelity_benchmark(
        model,
        h_state,
        n_spins=n_spins,
        t_steps=args.t_steps,
        t_max=args.t_max,
        dt=dt,
        apply_H_exact=apply_H_exact,
        state_count=args.state_benchmark_count,
        basis_family=args.state_benchmark_basis,
        sample_seed=args.state_benchmark_seed,
        alpha_chunk=args.state_benchmark_alpha_chunk,
        beta_chunk=args.state_benchmark_beta_chunk,
        max_x_spins=args.state_benchmark_max_x_spins,
    )
    save_json(output_dir / "state_benchmark.json", state_benchmark)
    sb_summary = state_benchmark["summary"]
    print(
        "[state benchmark] "
        f"evaluated={sb_summary['num_evaluated']}, "
        f"skipped={sb_summary['num_skipped']}, "
        f"avg_final_fid={sb_summary['avg_of_final_fidelity']}"
    )

summary = {
    "n_spins": n_spins,
    "ctx_tokens": int(args.ctx_tokens),
    "embed_dim": int(args.d_model),
    "mlp_mult": int(args.mlp_mult),
    "start_lr": float(args.start_lr),
    "min_lr": float(args.min_lr),
    "decay_rate": float(args.decay_rate),
    "decay_steps": int(args.decay_steps),
    "steps": int(args.steps),
    "anchor_pretrain_steps": int(args.anchor_pretrain_steps),
    "t_steps": int(args.t_steps),
    "t_max": float(args.t_max),
    "batch_size": int(args.batch_size),
    "num_time_points": int(args.num_time_points),
    "num_mc_samples": int(args.num_mc_samples),
    "time_curriculum": bool(args.time_curriculum),
    "time_curriculum_start_frac": float(args.time_curriculum_start_frac),
    "time_curriculum_power": float(args.time_curriculum_power),
    "phase_mode": args.phase_mode,
    "gauge_centering": args.gauge_centering,
    "self_kv_cache": bool(args.self_kv_cache),
    "best_loss": float(best_loss),
    "last_loss": float(loss_hist[-1]) if loss_hist else None,
    "benchmark": benchmark,
    "state_benchmark": None if state_benchmark is None else state_benchmark["summary"],
}
save_json(output_dir / "summary.json", summary)
eqx.tree_serialise_leaves(output_dir / "trained_model.eqx", model)
print("\nFinal summary")
print(f"Wrote results to: {output_dir}")
print(f"Checkpoint written to: {checkpoint_path}")

JAX is using: [CudaDevice(id=0)]
System: 4x4 = 16 spins
--- Training Propagator NOQS Model (TFIM + longitudinal field, OBC) ---
M0 mode: random_unit_rows
Phase mode: raw
Gauge centering: per_time
Score function weight: 1.0
Self-KV cache: False
Context dynamics: direct FNO Mdot with simpson integration
Site order: snake
Time curriculum: disabled
Pre-training identity anchor...


100%|██████████| 500/500 [00:14<00:00, 35.48it/s] 


TDVP Training...


Loss: 0.00197: 100%|██████████| 120000/120000 [4:52:07<00:00,  6.85it/s]  


Training Complete. Best loss achieved: 0.000567

Final summary
Wrote results to: /home/zqi/learn_propagator/learn_propagator_run
Checkpoint written to: /home/zqi/learn_propagator/learn_propagator_latest_checkpoint_4x4.pkl
